# Orlando Housing Fair Market Value Prediction Model

**Objective:** Build an XGBoost Regressor to predict fair market value (FMV) for Orlando properties.

**Pipeline:**
1. Data Loading & Cleaning
2. Feature Engineering
3. Train/Test Split
4. Hyperparameter Tuning (GridSearchCV)
5. Cross-Validation Analysis
6. Test Set Evaluation
7. Feature Importance
8. Save Model & Prediction Function

## 0. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

print("✓ Libraries loaded successfully")

ModuleNotFoundError: No module named 'shap'

## 1. Data Loading & Cleaning

In [ ]:
def load_and_clean_data(filepath):
    """Load data and handle formatting issues."""
    df = pd.read_csv(filepath)
    
    # Strip whitespace from column names
    df.columns = df.columns.str.strip()
    
    # Columns that need numeric cleaning (have $, commas, spaces)
    currency_cols = ['SALE_PRC', 'LND_SQFOOT', 'TOT_LVG_AREA', 'RAIL_DIST', 
    'OCEAN_DIST', 'WATER_DIST', 'CNTR_DIST', 'SUBCNTR_DI', 'HWY_DIST']
    
    for col in currency_cols:
        df[col] = (df[col].astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
        .astype(float))
    
    # Drop ID column - no predictive value
    df = df.drop(columns=['PARCELNO'])
    
    return df

In [ ]:
# Load the data
df = load_and_clean_data('..\data\orlando-housing-data\orlando-housing-data.csv')

print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Target Range: ${df['SALE_PRC'].min():,.0f} - ${df['SALE_PRC'].max():,.0f}")
print(f"Target Median: ${df['SALE_PRC'].median():,.0f}")
df.head()

In [ ]:
# Quick data summary
df.describe()

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

## 2. Exploratory Data Analysis

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original scale
axes[0].hist(df['SALE_PRC'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Sale Price ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Sale Price Distribution (Original)')
axes[0].axvline(df['SALE_PRC'].median(), color='red', linestyle='--', label=f'Median: ${df["SALE_PRC"].median():,.0f}')
axes[0].legend()

# Log scale
axes[1].hist(np.log1p(df['SALE_PRC']), bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('Log(Sale Price)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Sale Price Distribution (Log Transformed)')

plt.tight_layout()
plt.show()

print("→ Log transform normalizes the right-skewed distribution")

In [ ]:
# Correlation with target
correlations = df.corr()['SALE_PRC'].drop('SALE_PRC').sort_values(ascending=False)

plt.figure(figsize=(10, 8))
colors = ['green' if x > 0 else 'red' for x in correlations]
correlations.plot(kind='barh', color=colors)
plt.xlabel('Correlation with SALE_PRC')
plt.title('Feature Correlations with Target')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Key feature relationships
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Living Area vs Price
axes[0, 0].scatter(df['TOT_LVG_AREA'], df['SALE_PRC'], alpha=0.3, s=10)
axes[0, 0].set_xlabel('Total Living Area (sqft)')
axes[0, 0].set_ylabel('Sale Price ($)')
axes[0, 0].set_title(f'Living Area vs Price (r={df["TOT_LVG_AREA"].corr(df["SALE_PRC"]):.3f})')

# Land SqFoot vs Price
axes[0, 1].scatter(df['LND_SQFOOT'], df['SALE_PRC'], alpha=0.3, s=10, color='orange')
axes[0, 1].set_xlabel('Land Square Footage')
axes[0, 1].set_ylabel('Sale Price ($)')
axes[0, 1].set_title(f'Land Size vs Price (r={df["LND_SQFOOT"].corr(df["SALE_PRC"]):.3f})')

# Age vs Price
axes[1, 0].scatter(df['age'], df['SALE_PRC'], alpha=0.3, s=10, color='green')
axes[1, 0].set_xlabel('Property Age (years)')
axes[1, 0].set_ylabel('Sale Price ($)')
axes[1, 0].set_title(f'Age vs Price (r={df["age"].corr(df["SALE_PRC"]):.3f})')

# Structure Quality vs Price
df.boxplot(column='SALE_PRC', by='structure_quality', ax=axes[1, 1])
axes[1, 1].set_xlabel('Structure Quality')
axes[1, 1].set_ylabel('Sale Price ($)')
axes[1, 1].set_title('Structure Quality vs Price')
plt.suptitle('')

plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
def engineer_features(df):
    """Create derived features to improve model performance."""
    df = df.copy()
    
    # --- Ratio Features ---
    # Lot utilization: how much of the land is built on
    df['lot_utilization'] = df['TOT_LVG_AREA'] / df['LND_SQFOOT']
    
    # Living area per year of age (newer homes tend to be larger)
    df['area_per_age'] = df['TOT_LVG_AREA'] / (df['age'] + 1)  # +1 to avoid div by zero
    
    # Special features as percentage of typical home value
    df['spec_feat_ratio'] = df['SPEC_FEAT_VAL'] / (df['TOT_LVG_AREA'] * 200)  # ~$200/sqft baseline
    
    # --- Distance Interaction Features ---
    # Combined urban accessibility score (closer to center & subcenter = higher)
    df['urban_access'] = 1 / (np.log1p(df['CNTR_DIST']) + np.log1p(df['SUBCNTR_DI']))
    
    # Water proximity premium indicator
    df['near_water'] = (df['WATER_DIST'] < 1000).astype(int)
    
    # Highway convenience (close but not too close)
    df['hwy_convenience'] = np.where(
        (df['HWY_DIST'] > 500) & (df['HWY_DIST'] < 5000), 1, 0
    )
    
    # --- Cyclical Encoding for Month ---
    # Captures seasonality in housing market
    df['month_sin'] = np.sin(2 * np.pi * df['month_sold'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month_sold'] / 12)
    
    # --- Location Features ---
    # Distance from geographic center of dataset (captures "centrality")
    lat_center = 28.5383 # Hardcoded Downtown Orlando Latitude
lon_center = -81.3792 # Hardcoded Downtown Orlando Longitude
    df['dist_from_center'] = np.sqrt((df['LATITUDE'] - lat_center)**2 + (df['LONGITUDE'] - lon_center)**2)
    
    # --- Log Transforms for Skewed Distance Features ---
    distance_cols = ['RAIL_DIST', 'OCEAN_DIST', 'WATER_DIST', 'CNTR_DIST', 'SUBCNTR_DI', 'HWY_DIST']
    for col in distance_cols:
        df[f'{col}_log'] = np.log1p(df[col])
    
    # --- Binary: Has Special Features ---
    df['has_spec_feat'] = (df['SPEC_FEAT_VAL'] > 0).astype(int)
    
    # --- Structure Quality Squared (captures non-linear quality premium) ---
    df['quality_sq'] = df['structure_quality'] ** 2
    
    return df

In [ ]:
# Apply feature engineering
df_engineered = engineer_features(df)

print(f"Original features: {len(df.columns)}")
print(f"After engineering: {len(df_engineered.columns)}")
print(f"New features added: {len(df_engineered.columns) - len(df.columns)}")
print("\nNew feature columns:")
print([col for col in df_engineered.columns if col not in df.columns])

## 4. Prepare Data for Modeling

In [ ]:
# Separate features and target
X = df_engineered.drop(columns=['SALE_PRC'])
y = np.log1p(df_engineered['SALE_PRC'])  # Log transform target

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature list ({len(X.columns)} features):")
print(X.columns.tolist())

In [ ]:
# Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train):,} samples")
print(f"Test set: {len(X_test):,} samples")

## 5. Hyperparameter Tuning (GridSearchCV)

In [ ]:
# Define base model
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

# Parameter grid
param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.8, 0.9],
}

total_combinations = np.prod([len(v) for v in param_grid.values()])
print(f"Total parameter combinations to search: {total_combinations}")

In [ ]:
%%time

# Run GridSearchCV
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

In [ ]:
# Best parameters
print("="*60)
print("BEST HYPERPARAMETERS")
print("="*60)
print(f"Best CV RMSE (log scale): {-grid_search.best_score_:.4f}")
print("\nBest Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

In [ ]:
# Get best model
best_model = grid_search.best_estimator_
print(f"Best model: {best_model}")

## 6. Cross-Validation Analysis

In [ ]:
def cross_validation_analysis(model, X, y, cv_folds=5):
    """Detailed cross-validation analysis with multiple metrics."""
    
    kfold = KFold(n_splits=cv_folds, shuffle=True, random_state=42)
    
    # Store results per fold
    fold_results = {
        'fold': [],
        'rmse': [],
        'mae': [],
        'r2': [],
        'mape': []
    }
    
    for fold, (train_idx, val_idx) in enumerate(kfold.split(X), 1):
        X_train_fold = X.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]
        
        # Clone and fit model
        fold_model = xgb.XGBRegressor(**model.get_params())
        fold_model.fit(X_train_fold, y_train_fold, verbose=False)
        
        # Predictions (in log space)
        y_pred_log = fold_model.predict(X_val_fold)
        
        # Convert back to original scale for interpretable metrics
        y_val_orig = np.expm1(y_val_fold)
        y_pred_orig = np.expm1(y_pred_log)
        
        # Calculate metrics
        rmse = np.sqrt(mean_squared_error(y_val_orig, y_pred_orig))
        mae = mean_absolute_error(y_val_orig, y_pred_orig)
        r2 = r2_score(y_val_orig, y_pred_orig)
        mape = np.mean(np.abs((y_val_orig - y_pred_orig) / y_val_orig)) * 100
        
        fold_results['fold'].append(fold)
        fold_results['rmse'].append(rmse)
        fold_results['mae'].append(mae)
        fold_results['r2'].append(r2)
        fold_results['mape'].append(mape)
    
    return pd.DataFrame(fold_results)

In [ ]:
# Run CV analysis
cv_results = cross_validation_analysis(best_model, X_train, y_train, cv_folds=5)
cv_results

In [ ]:
# CV Summary
print("="*60)
print("CROSS-VALIDATION SUMMARY (5-Fold)")
print("="*60)
print(f"RMSE:  ${cv_results['rmse'].mean():,.0f} ± ${cv_results['rmse'].std():,.0f}")
print(f"MAE:   ${cv_results['mae'].mean():,.0f} ± ${cv_results['mae'].std():,.0f}")
print(f"R²:    {cv_results['r2'].mean():.4f} ± {cv_results['r2'].std():.4f}")
print(f"MAPE:  {cv_results['mape'].mean():.2f}% ± {cv_results['mape'].std():.2f}%")

In [ ]:
# Visualize CV results
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

metrics = ['rmse', 'mae', 'r2', 'mape']
titles = ['RMSE ($)', 'MAE ($)', 'R² Score', 'MAPE (%)']
colors = ['steelblue', 'coral', 'seagreen', 'mediumpurple']

for ax, metric, title, color in zip(axes, metrics, titles, colors):
    ax.bar(cv_results['fold'], cv_results[metric], color=color, alpha=0.7)
    ax.axhline(cv_results[metric].mean(), color='red', linestyle='--', label='Mean')
    ax.set_xlabel('Fold')
    ax.set_ylabel(title)
    ax.set_title(f'{title} by Fold')
    ax.legend()

plt.tight_layout()
plt.show()

## 7. Test Set Evaluation

In [ ]:
# Predictions on test set
y_pred_log = best_model.predict(X_test)

# Convert back to original scale
y_test_orig = np.expm1(y_test)
y_pred_orig = np.expm1(y_pred_log)

# Calculate metrics
rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
mae = mean_absolute_error(y_test_orig, y_pred_orig)
r2 = r2_score(y_test_orig, y_pred_orig)
mape = np.mean(np.abs((y_test_orig - y_pred_orig) / y_test_orig)) * 100

print("="*60)
print("TEST SET EVALUATION")
print("="*60)
print(f"RMSE:  ${rmse:,.0f}")
print(f"MAE:   ${mae:,.0f}")
print(f"R²:    {r2:.4f}")
print(f"MAPE:  {mape:.2f}%")

In [ ]:
# Prediction accuracy buckets
errors = np.abs(y_test_orig - y_pred_orig)
pct_errors = np.abs((y_test_orig - y_pred_orig) / y_test_orig) * 100

print("\nPrediction Accuracy:")
print(f"  Within $25K:   {(errors <= 25000).mean() * 100:.1f}%")
print(f"  Within $50K:   {(errors <= 50000).mean() * 100:.1f}%")
print(f"  Within $100K:  {(errors <= 100000).mean() * 100:.1f}%")
print(f"  Within 5%:     {(pct_errors <= 5).mean() * 100:.1f}%")
print(f"  Within 10%:    {(pct_errors <= 10).mean() * 100:.1f}%")
print(f"  Within 15%:    {(pct_errors <= 15).mean() * 100:.1f}%")

In [ ]:
# Actual vs Predicted Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(y_test_orig, y_pred_orig, alpha=0.3, s=10)
axes[0].plot([y_test_orig.min(), y_test_orig.max()], 
             [y_test_orig.min(), y_test_orig.max()], 
             'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Sale Price ($)')
axes[0].set_ylabel('Predicted Sale Price ($)')
axes[0].set_title(f'Actual vs Predicted (R² = {r2:.4f})')
axes[0].legend()

# Residual distribution
residuals = y_test_orig - y_pred_orig
axes[1].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Residual Distribution (Mean: ${residuals.mean():,.0f})')

plt.tight_layout()
plt.show()

In [ ]:
# Residuals vs Predicted (check for heteroscedasticity)
plt.figure(figsize=(10, 5))
plt.scatter(y_pred_orig, residuals, alpha=0.3, s=10)
plt.axhline(0, color='red', linestyle='--', lw=2)
plt.xlabel('Predicted Sale Price ($)')
plt.ylabel('Residual ($)')
plt.title('Residuals vs Predicted Values')
plt.tight_layout()
plt.show()

## 8. Feature Importance Analysis

In [ ]:
# Extract feature importance
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

importance_df.head(15)

In [ ]:
# Feature importance plot
plt.figure(figsize=(10, 10))
top_n = 20
top_features = importance_df.head(top_n)

plt.barh(range(top_n), top_features['importance'].values, align='center')
plt.yticks(range(top_n), top_features['feature'].values)
plt.xlabel('Feature Importance')
plt.ylabel('Feature')
plt.title(f'Top {top_n} Feature Importance (XGBoost)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance summary
print("="*60)
print("TOP 10 MOST IMPORTANT FEATURES")
print("="*60)
for i, row in importance_df.head(10).iterrows():
    bar = '█' * int(row['importance'] * 50)
    print(f"{row['feature']:<25} {row['importance']:.4f} {bar}")

## 9. Save Model & Artifacts

In [ ]:
# Save model
best_model.save_model('orlando_fmv_xgboost_model.json')
print("✓ Model saved: orlando_fmv_xgboost_model.json")

# Save feature importance
importance_df.to_csv('feature_importance.csv', index=False)
print("✓ Feature importance saved: feature_importance.csv")

# Save CV results
cv_results.to_csv('cv_results.csv', index=False)
print("✓ CV results saved: cv_results.csv")

# Save feature list (needed for prediction)
feature_list = X.columns.tolist()
pd.Series(feature_list).to_csv('feature_list.csv', index=False, header=False)
print("✓ Feature list saved: feature_list.csv")

## 10. Prediction Function for New Properties

In [ ]:
def predict_fair_market_value(model, feature_names, property_data):
    """
    Predict FMV for a new property.
    
    Parameters:
    -----------
    model : trained XGBoost model
    feature_names : list of feature names in correct order
    property_data : dict with property attributes
    
    Returns:
    --------
    float : Predicted fair market value in USD
    """
    # Create DataFrame with single row
    df = pd.DataFrame([property_data])
    
    # Apply same feature engineering
    df = engineer_features(df)
    
    # Ensure columns are in correct order
    df = df[feature_names]
    
    # Predict (model returns log-transformed value)
    pred_log = model.predict(df)[0]
    pred_value = np.expm1(pred_log)
    
    return pred_value

In [ ]:
# Example prediction
example_property = {
    'LATITUDE': 28.55,
    'LONGITUDE': -81.35,
    'LND_SQFOOT': 8500,
    'TOT_LVG_AREA': 2000,
    'SPEC_FEAT_VAL': 10000,
    'RAIL_DIST': 5000,
    'OCEAN_DIST': 55000,
    'WATER_DIST': 1500,
    'CNTR_DIST': 40000,
    'SUBCNTR_DI': 8000,
    'HWY_DIST': 3000,
    'age': 15,
    'avno60plus': 0,
    'month_sold': 6,
    'structure_quality': 4.0
}

predicted_fmv = predict_fair_market_value(best_model, feature_list, example_property)

print("="*60)
print("EXAMPLE PREDICTION")
print("="*60)
print("\nProperty Attributes:")
for key, value in example_property.items():
    print(f"  {key}: {value}")
print(f"\n>>> Predicted Fair Market Value: ${predicted_fmv:,.0f}")

## 11. Load Model & Predict (For Future Use)

In [ ]:
# How to load the saved model later
def load_model_and_predict(model_path, feature_list_path, property_data):
    """
    Load saved model and make predictions.
    
    Usage:
    ------
    predicted_value = load_model_and_predict(
        'orlando_fmv_xgboost_model.json',
        'feature_list.csv',
        property_dict
    )
    """
    # Load model
    loaded_model = xgb.XGBRegressor()
    loaded_model.load_model(model_path)
    
    # Load feature list
    feature_names = pd.read_csv(feature_list_path, header=None)[0].tolist()
    
    # Predict
    return predict_fair_market_value(loaded_model, feature_names, property_data)

print("✓ load_model_and_predict() function defined for future use")

---

## Summary

| Metric | Value |
|--------|-------|
| **Dataset Size** | 30,000 properties |
| **Features (Original)** | 16 |
| **Features (Engineered)** | 28 |
| **Best Model** | XGBoost Regressor |
| **Cross-Validation** | 5-Fold |

### Files Saved:
- `orlando_fmv_xgboost_model.json` - Trained model
- `feature_importance.csv` - Feature rankings
- `cv_results.csv` - Cross-validation metrics
- `feature_list.csv` - Feature names for prediction